In [ ]:
# 🟠 LEVEL 5: Input Construction (VERY
# IMPORTANT)
# 👉 This is your multi-signal fusion


In [ ]:
# 39. Generate:
# ● RGB
# ● FFT
# ● Noise
# ● ELA
# for ONE image
# 40. Convert all to same shape (224×224)
# 41. Stack into multi-channel tensor
# 👉 Target:
# (6, 224, 224)
# 42. Visualize each channel separately
import cv2
import numpy as np
import torch
from scipy.fftpack import fft2, fftshift

def get_multi_signal_tensor(image_path, target_size=(224, 224)):
    # 1. Load and Resize Base Image
    img = cv2.imread(image_path)
    img = cv2.resize(img, target_size)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    # --- SIGNAL 1: RGB (3 Channels) ---
    rgb_channels = img_rgb.astype(np.float32) / 255.0

    # --- SIGNAL 2: FFT (1 Channel) ---
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    f_transform = fft2(gray)
    f_shift = fftshift(f_transform)
    magnitude_spectrum = np.log(np.abs(f_shift) + 1)
    fft_channel = cv2.normalize(magnitude_spectrum, None, 0, 1, cv2.NORM_MINMAX)

    # --- SIGNAL 3: Noise Residue (1 Channel) ---
    # Using a median filter subtraction to extract high-frequency noise
    denoised = cv2.medianBlur(gray, 3)
    noise_residue = cv2.absdiff(gray, denoised)
    noise_channel = cv2.normalize(noise_residue, None, 0, 1, cv2.NORM_MINMAX)

    # --- SIGNAL 4: ELA (1 Channel) ---
    # Resave at lower quality and find the difference
    path_temp = "temp_ela.jpg"
    cv2.imwrite(path_temp, img, [cv2.IMWRITE_JPEG_QUALITY, 90])
    img_low = cv2.imread(path_temp)
    ela_diff = cv2.absdiff(img, img_low)
    ela_gray = cv2.cvtColor(ela_diff, cv2.COLOR_BGR2GRAY)
    ela_channel = cv2.normalize(ela_gray, None, 0, 1, cv2.NORM_MINMAX)

    # 2. Stack into (6, 224, 224)
    # Reorganize RGB from (H, W, 3) to (3, H, W)
    rgb_stack = np.transpose(rgb_channels, (2, 0, 1))
    
    # Expand dims for single channels (1, 224, 224)
    fft_stack = fft_channel[np.newaxis, ...]
    noise_stack = noise_channel[np.newaxis, ...]
    ela_stack = ela_channel[np.newaxis, ...]

    # Final Fusion
    fusion_tensor = np.concatenate([rgb_stack, fft_stack, noise_stack, ela_stack], axis=0)
    
    return torch.from_numpy(fusion_tensor), [rgb_channels, fft_channel, noise_channel, ela_channel]

# Execute
tensor, visuals = get_multi_signal_tensor("your_image.jpg")
print(f"Final Tensor Shape: {tensor.shape}")